# Hour 3 · Letters as social networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/3b_letter_networks.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/main?labpath=notebooks%2F3b_letter_networks.ipynb)


**Goal:** turn the corpus of **letters** into a graph of correspondents.

**Needs:** `networkx`. Ugaritic letters open with a fixed address formula: **`tḥm X`** = 'message of X' (sender) and **`l Y rgm`** = 'to Y, say' (recipient).

*Reading:* `docs/06-letters-networks.md`.

## 0. Setup


In [1]:
# === SETUP — run me first (works in Google Colab, Binder, and locally) ===
import os, sys, subprocess

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # only umap-learn is usually missing. Install just that (fast).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)

# make data/loader.py importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))
from data.loader import load_texts

texts = load_texts()     # 1st Colab run downloads the CUC JSONL from HuggingFace, then caches it
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")

[loader] Loaded 278 CUC tablets, 25477 word tokens (source: AlexWalhai/cuc JSONL cache, licence: CC BY-NC 4.0).
Loaded 278 tablets — genre of the first one: myth


## 1. Parse (sender → recipient) from the address formula

In [2]:
# particles that are not names (drop them if the formula parse grabs one)
STOP = {"l", "w", "b", "k", "ʿm", "il", "ilm"}

def parse_letter(t):
    toks = t["tokens"]
    sender = recipient = None
    for i, w in enumerate(toks[:-1]):
        if w == "tḥm" and sender is None:
            sender = toks[i+1]
        if w == "l" and recipient is None:
            recipient = toks[i+1]
    return sender, recipient

edges = []
for t in texts:
    if t["genre"] != "letter": continue
    s, r = parse_letter(t)
    if s and r and s != r and s not in STOP and r not in STOP:
        edges.append((s, r, t["ktu"]))
print(len(edges), "edges parsed")
edges[:12]

40 edges parsed


[('iwrḏr', 'plsy', '2.10'),
 ('skn', 'mlkt', '2.101'),
 ('mlk', 'skn', '2.102'),
 ('ʿṯty', 'urtn', '2.104'),
 ('aġltṯb', 'xxxyn', '2.105'),
 ('tlmyn', 'umy', '2.11'),
 ('tlmyn', 'mlkt', '2.12'),
 ('mlk', 'mlkt', '2.13'),
 ('ixwrḏn', 'iwrpḫn', '2.14'),
 ('tlmyn', 'ṯryl', '2.16'),
 ('mlk', 'mlkt', '2.21'),
 ('illḏr', 'mlkt', '2.24')]

## 2. Build the directed correspondence graph

In [3]:
import networkx as nx
G = nx.DiGraph()
for s, r, k in edges:
    G.add_edge(s, r, ktu=k)
print("people:", G.number_of_nodes(), " letters:", G.number_of_edges())

people: 43  letters: 34


## 3. Draw it — an interactive, draggable network

Static layouts can't show a 21-person hub legibly. So here it's a **live force-directed graph** (vis-network): **drag any node**, scroll to zoom, and let the physics spread the clusters apart. **Hover** a node for who it wrote to and received from; node size = number of ties; colour = which separate **circle** it belongs to. The real shape emerges on its own — one **royal hub** plus several **two-person exchanges** drifting free.

In [4]:
import json
from IPython.display import HTML, display

comp_of = {n: i for i, comp in enumerate(
    sorted(nx.weakly_connected_components(G), key=len, reverse=True)) for n in comp}
PAL = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
       "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#aec7e8", "#ffbb78"]

nodes = [{"id": n, "label": n, "value": int(G.degree(n)), "color": PAL[comp_of[n] % len(PAL)],
          "title": f"{n}: sent {G.out_degree(n)}, received {G.in_degree(n)}"} for n in G.nodes()]
links = [{"from": s, "to": r, "title": f"KTU {d.get('ktu', '')}"} for s, r, d in G.edges(data=True)]

doc = """<!doctype html><html><head>
<script src="https://unpkg.com/vis-network@9.1.9/standalone/umd/vis-network.min.js"></script>
<style>html,body,#net{margin:0;height:100%%} #net{height:660px}</style></head>
<body><div id="net"></div><script>
const data = {nodes: new vis.DataSet(%s), edges: new vis.DataSet(%s)};
new vis.Network(document.getElementById("net"), data, {
  nodes: {shape: "dot", scaling: {min: 8, max: 45}, font: {size: 15}},
  edges: {arrows: "to", color: {color: "#c0c0c0"}, smooth: {type: "continuous"}},
  physics: {barnesHut: {gravitationalConstant: -9000, springLength: 120, avoidOverlap: 0.3},
            stabilization: {iterations: 250}},
  interaction: {hover: true, dragNodes: true, zoomView: true, tooltipDelay: 80}});
</script></body></html>""" % (json.dumps(nodes, ensure_ascii=False), json.dumps(links, ensure_ascii=False))

display(HTML('<iframe srcdoc="{}" width="100%" height="690" style="border:1px solid #eee">'
             '</iframe>'.format(doc.replace('"', "&quot;"))))
print("Drag the nodes; hover for sent/received; scroll to zoom.")

/Users/alexandersosnovschenko/projects/summer_school_2026/Antiquity Studies Summer School/.venv/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Drag the nodes; hover for sent/received; scroll to zoom.


## 4. Who is central? Three measures of importance

"Central" means different things — three standard measures, each answering a different question:

- **Degree centrality** — *how many ties?* The share of all other people a node is directly linked to (letters sent + received). High degree = a **busy correspondent**.
- **Betweenness centrality** — *how often a bridge?* The fraction of shortest paths between other pairs that run through this node. High betweenness = a **broker / go-between** joining otherwise separate parties.
- **Eigenvector centrality** — *connected to whom?* Not just *many* ties, but ties to *well-connected* others — influence by association. High eigenvector = sitting at the **heart of the connected core**. (Computed on the largest connected component; the isolated dyads score 0.)

In [5]:
import pandas as pd

deg = nx.degree_centrality(G)
bet = nx.betweenness_centrality(G)                                   # directed shortest paths
core = max(nx.weakly_connected_components(G), key=len)               # the royal hub
eig_core = nx.eigenvector_centrality_numpy(G.subgraph(core).to_undirected())
eig = {n: eig_core.get(n, 0.0) for n in G.nodes()}                   # 0 outside the core

cent = pd.DataFrame({"degree": deg, "betweenness": bet, "eigenvector": eig}).round(3)
print("Top correspondents — the three centralities side by side:")
cent.sort_values("degree", ascending=False).head(10)

Top correspondents — the three centralities side by side:


,degree,betweenness,eigenvector
mlk,0.238,0.020,0.555
mlkt,0.167,0.013,0.475
tlmyn,0.119,0.000,0.264
urtn,0.071,0.000,0.161
skn,0.048,0.000,0.292
ʿbdk,0.048,0.000,0.171
ṯryl,0.048,0.000,0.232
rb,0.048,0.000,0.157
ʿṯty,0.024,0.000,0.046
plsy,0.024,0.000,0.000


**Read across the columns.** The **king (mlk)** and **queen (mlkt)** lead on all three — and are the *only* nodes with non-zero **betweenness**, the sole brokers linking sub-groups. Watch where the measures **disagree**: *skn* outranks *tlmyn* on **eigenvector** despite fewer ties — embeddedness in the royal core, not sheer volume. That gap between *degree* and *eigenvector* is exactly why one number is never enough.

## 5. Discussion
- The three measures **agree at the very top** (the royal pair *mlk* / *mlkt*) but **disagree just below it**: *skn* (a governor) is more *eigenvector*-central than *tlmyn* despite **fewer** ties, because *skn* writes straight to the king. Relying on one measure alone would mislead — that is why we compute all three.
- Only **two** nodes have non-zero **betweenness**: the network is a **hub-and-spoke** around the crown plus disconnected dyads, not a richly cross-linked web.
- Central nodes are often **titles / relations** (*mlk* king, *umy* 'my mother/lady') rather than unique persons — a real interpretive caveat. **Fragmentary preservation** also biases the graph (a missing edge ≠ no relationship), and explains the many isolated two-person components.

> **Production version:** UgaritLab `build_name_graph.py` builds a co-occurrence graph of named entities (PN/DN/TN/RN) from DULAT matched in UDB tablets — far richer than this parser.